In [3]:
!pip install "black[jupyter]" flake8 nbqa pre-commit
!pip install pytorch-lightning
!pip install torchmetrics
!pip install -U "wandb>=0.12.10"
!pip install ipytest
import torch
import torch.nn as nn
import pytorch_lightning as pl
import torch.nn.functional as f
import torchmetrics
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import WandbLogger
import doctest
import ipytest
import os

In [4]:
%%writefile .pre-commit-config.yaml
repos:
  - repo: https://github.com/nbQA-dev/nbQA
    rev: 1.7.0
    hooks:
      - id: nbqa-black
      - id: nbqa-flake8

Overwriting .pre-commit-config.yaml


In [5]:
transform = transforms.Compose([transforms.ToTensor()])

# Download datasets (EMNIST requires split parameter, 'byclass' has 62 classes)
train_dataset = datasets.EMNIST(root='./data', split='byclass', train=True, download=True, transform=transform)
val_dataset = datasets.EMNIST(root='./data', split='byclass', train=False, download=True, transform=transform)

train_data = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=os.cpu_count())
val_data = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=os.cpu_count())

# Callbacks
check_point_callback = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="smartest_model_checkpoint"
)
STOP_Early_callback = EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min"
)
#Wandb Loggers
wandb_logger = WandbLogger(project="my_fsdl_project", name="experiment_1")


100%|██████████| 562M/562M [00:08<00:00, 65.3MB/s]


In [ ]:
import wandb
import pytorch_lightning as pl
# Force lightning to re-detect wandb if it cached a failure previously
import pytorch_lightning.loggers.wandb as pl_wandb
pl_wandb._WANDB_AVAILABLE = True

import torch
import torch.nn as nn
import torchmetrics
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import WandbLogger

class ModularCNN(pl.LightningModule):
  def __init__(self):
    super().__init__()

    # block 1:
    self.block1 = nn.Sequential(
        nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )

    # block 2:
    self.block2 = nn.Sequential(
        nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )

    self.dropoff = nn.Dropout(0.25)
    # With 28x28 input:
    # Conv2d(32, 3x3) -> 26x26. MaxPool2d(2x2) -> 13x13
    # Conv2d(64, 3x3) -> 11x11. MaxPool2d(2x2) -> 5x5
    # Features: 64 * 5 * 5 = 1600
    self.output = nn.Linear(in_features=1600, out_features=62) # EMNIST byclass has 62 classes
    self.loss = nn.CrossEntropyLoss()
    self.accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=62)

  def forward(self, x):
    x = self.block1(x)
    x = self.block2(x)
    x = torch.flatten(x, 1)
    x = self.dropoff(x)
    x = self.output(x)
    return x

  def training_step(self, batch, batch_idx):
    x, y = batch
    prediction = self(x)
    loss_output = self.loss(prediction, y)
    self.log("train_loss", loss_output, prog_bar=True)
    return loss_output

  def validation_step(self, batch, batch_idx):
    x, y = batch
    prediction = self(x)
    acc = self.accuracy(prediction, y)
    loss_output = self.loss(prediction, y)
    self.log("val_loss", loss_output, prog_bar=True)
    self.log("val_acc", acc, prog_bar=True)
    return loss_output

  def configure_optimizers(self):
    return torch.optim.Adam(self.parameters(), lr=0.01)


if __name__ == "__main__":
  model = ModularCNN()
  Trainer = pl.Trainer(callbacks=[check_point_callback, STOP_Early_callback], max_epochs=5, logger=wandb_logger)
  Trainer.fit(model, train_data, val_data)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name     ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ block1   │ Sequential         │    384 │ train │     0 │
│ 1 │ block2   │ Sequential         │ 18.6 K │ train │     0 │
│ 2 │ dropoff  │ Dropout            │      0 │ train │     0 │
│ 3 │ output   │ Linear             │ 99.3 K │ train │     0 │
│ 4 │ loss     │ CrossEntropyLoss   │      0 │ train │     0 │
│ 5 │ accuracy │ MulticlassAccuracy │      0 │ train │     0 │
└───┴──────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 118 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 118 K                                                                                                
Total estimated model params size (MB): 0.473                                                                      
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


In [ ]:
!git init
!git add .
!pre-commit run --all-files

Reinitialized existing Git repository in /content/.git/
nbqa-black...............................................................Passed
nbqa-flake8..............................................................Passed


In [ ]:
import torch

# 1. Load the best trained model from the checkpoint your ModelCheckpoint callback saved
checkpoint_path = check_point_callback.best_model_path
print(f"Loading best checkpoint from: {checkpoint_path}")

# Load weights and set to eval mode (disables dropout)
trained_model = ModularCNN.load_from_checkpoint(checkpoint_path)
trained_model.eval()

# 2. PyTorch Lightning's built-in conversion method (uses tracing under the hood)
# EMNIST is single-channel 28x28, so input shape is [batch_size, 1, 28, 28]
dummy_input = torch.randn(1, 1, 28, 28)

scripted_model = trained_model.to_torchscript(
    file_path="modular_cnn_torchscript.pt",
    method="trace",
    example_inputs=dummy_input
)

print("Model successfully compiled and saved to 'modular_cnn_torchscript.pt'!")

Loading best checkpoint from: ./my_fsdl_project/hinkr7uc/checkpoints/smartest_model_checkpoint.ckpt
Model successfully compiled and saved to 'modular_cnn_torchscript.pt'!


In [ ]:
import torch

# Load the standalone TorchScript binary on CPU
device = torch.device("cpu")
loaded_model = torch.jit.load("modular_cnn_torchscript.pt", map_location=device)
loaded_model.eval()

# Run a sample forward pass (1 image, 1 channel, 28x28)
sample_input = torch.randn(1, 1, 28, 28)
with torch.no_grad():
    output = loaded_model(sample_input)

print("Output shape:", output.shape)  # Should be [1, 62]
print("Predicted class:", torch.argmax(output, dim=1).item())

Output shape: torch.Size([1, 62])
Predicted class: 16


**Reasoning**:
Install the required packages for the FastAPI backend service: fastapi, uvicorn, python-multipart, and pillow.



In [ ]:
!pip install fastapi uvicorn python-multipart pillow

**Reasoning**:
Create the backend directory and implement the `main.py` script containing the FastAPI service according to the instructions. The EMNIST 'byclass' split has 62 classes, representing digits 0-9, uppercase letters A-Z, and lowercase letters a-z.



In [ ]:
import os

# Create backend directory
os.makedirs("backend", exist_ok=True)

# Map EMNIST byclass indices (0-61) to their characters
# 0-9: digits
# 10-35: uppercase letters (A-Z)
# 36-61: lowercase letters (a-z)
emnist_classes = (
    [str(i) for i in range(10)] +
    [chr(i) for i in range(65, 91)] +
    [chr(i) for i in range(97, 123)]
)

with open("backend/main.py", "w") as f:
    f.write('''import io
import torch
from fastapi import FastAPI, UploadFile, File
from PIL import Image
import torchvision.transforms as transforms

app = FastAPI(title="EMNIST Character Recognition Backend")

# Load the TorchScript model
MODEL_PATH = "../modular_cnn_torchscript.pt"
try:
    model = torch.jit.load(MODEL_PATH, map_location=torch.device("cpu"))
    model.eval()
    print(f"Successfully loaded model from {MODEL_PATH}")
except Exception as e:
    # Fallback in case of working directory variations
    MODEL_PATH = "modular_cnn_torchscript.pt"
    model = torch.jit.load(MODEL_PATH, map_location=torch.device("cpu"))
    model.eval()
    print(f"Successfully loaded model from fallback {MODEL_PATH}")

# Map of EMNIST 62 classes
emnist_classes = [str(i) for i in range(10)] + [chr(i) for i in range(65, 91)] + [chr(i) for i in range(97, 123)]

# Define preprocessing transform
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
])

@app.get("/")
def read_root():
    return {"status": "running", "model_loaded": model is not None}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes))

    # Preprocess image
    tensor = transform(image).unsqueeze(0)  # Shape: [1, 1, 28, 28]

    # Run inference
    with torch.no_grad():
        outputs = model(tensor)
        predicted_class_idx = torch.argmax(outputs, dim=1).item()

    predicted_char = emnist_classes[predicted_class_idx]

    return {
        "prediction_index": predicted_class_idx,
        "prediction_char": predicted_char
    }
''')

print("FastAPI backend file written successfully at backend/main.py.")

FastAPI backend file written successfully at backend/main.py.


**Reasoning**:
Install streamlit and create the frontend directory and interactive application file frontend/app.py.



In [ ]:
!pip install streamlit

import os

os.makedirs("frontend", exist_ok=True)

with open("frontend/app.py", "w") as f:
    f.write('''import streamlit as st
import requests
from PIL import Image
import io

st.set_page_config(page_title="EMNIST Character Recognizer", layout="centered")

st.title("📝 EMNIST Character Recognition Frontend")
st.write("Upload a handwritten character image to predict its class using our Deep Learning model.")

# Input field for backend URL
backend_url = st.sidebar.text_input("Backend API URL", value="http://localhost:8000/predict")

uploaded_file = st.file_uploader("Choose a character image...", type=["png", "jpg", "jpeg"])

if uploaded_file is not None:
    image = Image.open(uploaded_file)
    st.image(image, caption="Uploaded Image", use_container_width=True)

    if st.button("Identify Character"):
        with st.spinner("Analyzing image..."):
            try:
                # Convert PIL Image back to bytes for post request
                img_byte_arr = io.BytesIO()
                image.save(img_byte_arr, format=image.format if image.format else "PNG")
                img_byte_arr = img_byte_arr.getvalue()

                files = {"file": (uploaded_file.name, img_byte_arr, "image/png")}
                response = requests.post(backend_url, files=files, timeout=10)

                if response.status_code == 200:
                    result = response.json()
                    prediction_char = result.get("prediction_char", "Unknown")
                    prediction_idx = result.get("prediction_index", -1)

                    st.success(f"### Prediction: {prediction_char}")
                    st.info(f"Model Class Index: {prediction_idx}")
                else:
                    st.error(f"Backend returned error status code: {response.status_code}")
            except Exception as e:
                st.error(f"Failed to connect to the backend: {e}")
''')

print("Streamlit frontend file written successfully at frontend/app.py.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 119.6 MB/s eta 0:00:00
Streamlit frontend file written successfully at frontend/app.py.


# Task
Design and deploy a full-stack production application for character recognition using the compiled TorchScript model `"modular_cnn_torchscript.pt"` from the EMNIST dataset. The system must include:
1. A **FastAPI Backend Service** that loads `"modular_cnn_torchscript.pt"` on CPU, accepts image uploads via a REST API endpoint (such as `/predict`), preprocesses the images to match the model's 1x28x28 input requirement, and returns the predicted character class.
2. An interactive **Streamlit Frontend Application** that enables users to upload handwritten character images, displays the uploaded image, sends a prediction request to the FastAPI backend, and visualizes the predicted results.
3. Production-ready **Docker Containerization** configurations (such as Dockerfiles) for both the FastAPI backend and Streamlit frontend apps to ensure reproducibility.
4. A comprehensive architectural and deployment guide summarizing local testing, continuous integration, and cloud deployment.

## Design FastAPI Backend Service

### Subtask:
Install FastAPI, Uvicorn, and Python-Multipart, then implement a Python backend script using FastAPI that loads the TorchScript model on CPU, preprocesses uploaded images to a 1x28x28 grayscale format, and returns predictions.


## Design Streamlit Frontend Application

### Subtask:
Install Streamlit, then write a Streamlit frontend script that allows users to upload images and queries the FastAPI backend for predictions.
